In [176]:
import numpy as np
import pandas as pd
import re

In [177]:
clues = pd.read_csv("xd/clues.tsv", sep = "\t", skiprows=[2627538])

In [178]:
clues = clues.dropna()

In [179]:
clues.head(10)

,pubid,year,answer,clue
0,7xw,2021,1138,THX ___ (George Lucas film referenced in many ...
1,7xw,2021,125243,Prime whose first three digits are p^q and who...
2,7xw,2021,158719,Multiple of XI (except not in Roman numerals; ...
3,7xw,2021,1812,Overture whose use of cannons is canon
4,7xw,2021,1IRON,"Not even God can hit the ___ --Lee Trevino, af..."
5,7xw,2021,1UP,Virtual life
6,7xw,2021,2255887,"Phone number that, on ""Scrubs"", could (aptly) ..."
7,7xw,2021,328509,69 cubed (nice)
8,7xw,2021,390000,Number that equals p^8 - p^4 for some prime p ...
9,7xw,2021,403403,"When halved (?!?), smallest semiprime whose pr..."


In [180]:
clues.sample(10)

,pubid,year,answer,clue
4247012,nyt,1989,PENAL,Kind of colony
398122,bg,2002,INTO,Really busy with
2574919,nw,2016,CAP,Mushroom part
5905776,up,2006,TINA,"Louise of ""Gilligan's Island"""
2781880,nw,2022,SGT,Police rank below lt.
5217755,nyt,2019,MANOR,Stately residence
1297032,lat,2002,URI,Canton divided by the Reuss River
3322971,nyt,1959,SUABLE,Subject to court action.
4962137,nyt,2011,WIE,Michelle on a fairway
1702810,lat,2016,ANAT,Sci. subject


In [181]:
clues.pubid.value_counts()

pubid
nyt       2443925
nw         840746
lat        832437
up         714350
usa        683388
cs         504736
wsj        392375
atc        168903
nys        147295
wap        118878
bg         110137
beq        109791
pk         101217
tny         60952
che         58650
club        55995
avc         42659
onion       24331
awm         22734
slate       22702
ss          18219
rnr         17912
gfp         16578
sl          13933
cca         13804
ross        12453
atl         10117
cc           9641
eltana       8833
7xw          4596
cht          3788
db           3722
mm           3562
tm           3535
upc          3447
mg           1452
tpfat        1179
nym           696
tmdcr         538
pzz            78
nyh            74
kc             43
nyw            31
Name: count, dtype: int64

In [182]:
clues_selected = clues[(clues.year >= 1995) & clues.pubid.isin(["nyt","usa","wsj","up"])]

In [183]:
x_row = re.compile("X+")

In [184]:
letter_row = re.compile("[A-Z]+")

In [185]:
x_clues = [bool(letter_row.fullmatch(str(s))) and not bool(x_row.fullmatch(str(s))) for s in clues_selected.answer]

In [186]:
clues_selected = clues_selected[x_clues]

In [187]:
clues_reference = re.compile(r"\d+( |-)([aA]cross|[dD]own)")

In [188]:
clues_selected = clues_selected[[not bool(clues_reference.findall(x)) for x in clues_selected.clue]]

In [189]:
clues_selected.sample(10)

,pubid,year,answer,clue
6188620,up,2017,HAH,Gotcha!
5191579,nyt,2018,PANG,Twinge of guilt
6510022,usa,2000,FEDS,Some agts.
4654073,nyt,2002,ILIE,First name in 1970's tennis
6009109,up,2010,NINETIETH,Notable birthday
4970177,nyt,2012,COMAS,Some are drug-induced
6741464,usa,2012,ASPER,___ your instructions . . .
5901244,up,2006,ROWAN,Martin's TV sidekick
5348925,nyt,2023,SMOG,Cause of low visibility in some cities
7077768,usa,2025,LIAR,"Pinocchio, when his nose is growing"


In [190]:
clues_selected = clues_selected.copy()

In [191]:
clues_selected["length"] = [len(str(x)) for x in clues_selected.answer]

In [192]:
clues_selected.answer.unique()

array(['AAA', 'AAH', 'AANDP', ..., 'XMASLIST', 'YELLOWBELLPEPPER',
       'YELLOWFINISH'], shape=(188219,), dtype=object)

In [193]:
clues_selected.sample(10)

,pubid,year,answer,clue,length
7575068,wsj,2025,BURRITOS,Canyon pack animals,8
6346340,up,2023,SAOPAULO,Largest city in the Southern Hemisphere,8
5766768,up,2001,WHERETHEHEARTIS,Billie Letts' book,15
6124458,up,2014,PIGIRON,Blast-furnace output,7
6958936,usa,2020,PAM,Cooking spray brand,3
6763549,usa,2013,ARMOR,Medieval museum exhibit,5
6723777,usa,2011,DIAS,First navigator to round the Cape of Good Hope,4
6912645,usa,2018,WREN,Tiny songbird,4
7028287,usa,2023,CASINO,Place for playing poker,6
4947157,nyt,2011,LIPO,"Way to slim down, for short",4


In [194]:
clues_selected["clue_count"] = [int(x) for x in clues_selected.groupby("answer").transform("count").clue]

In [195]:
clues_selected.sample(10)

,pubid,year,answer,clue,length,clue_count
4695374,nyt,2003,PAWN,Promotable piece,4,151
6373206,up,2024,SRI,___ Lanka,3,817
6794106,usa,2014,EMIR,Middle East chieftain,4,898
6834040,usa,2015,RAG,Aid for a maid,3,467
7393442,wsj,2017,SLEETIER,"More hazardous for driving, perhaps",8,2
5203370,nyt,2019,ALLRISE,Standing order,7,23
4868449,nyt,2009,ACTOR,Lineman?,5,338
5954882,up,2008,OVINE,Just like ewe?,5,228
6986282,usa,2021,ORATE,Speak grandly,5,714
4977418,nyt,2012,HANS,Brinker on skates,4,124


In [196]:
clues_selected = clues_selected[clues_selected.clue_count >= 10]

In [197]:
clues_selected.shape

(2407547, 6)

In [198]:
import tqdm

In [199]:
clues_ten = pd.concat([x.sample(10) for _,x in tqdm.tqdm(clues_selected.groupby("answer"))])

100%|███████████████████████████████████| 32475/32475 [00:10<00:00, 3004.65it/s]


In [200]:
clues_ten.sample(10)

,pubid,year,answer,clue,length,clue_count
4860766,nyt,2008,SAUSAGE,It may be chain-linked,7,25
5815151,up,2003,QUASH,Invalidate,5,22
5706164,up,1999,READ,Hit the books,4,444
5745313,up,2001,CUL,_____ de sac,3,53
4797578,nyt,2006,SLUNG,Like the stone that slew Goliath,5,24
6430059,usa,1996,DRIES,Helps with the dishes,5,58
4985348,nyt,2012,NOLOAD,Like some mutual funds,6,15
6713910,usa,2010,REPOT,Do a gardener's job,5,78
7433405,wsj,2019,EXEMPT,Off the hook,6,30
5931279,up,2007,SISSIES,Tomboys' counterparts,7,11


In [213]:
clues_ten["clue_id"] = clues_ten.groupby("answer")["clue"].rank("first").astype(int)

In [214]:
clues_clean = pd.pivot(clues_ten, columns="clue_id", index="answer", values = "clue")

In [215]:
clues_clean.columns = [f"clue_{n+1}" for n in range(10)]

In [216]:
clues_clean = clues_clean.reset_index()

In [217]:
clues_clean = clues_clean.reset_index()

In [218]:
clues_clean.sample()

,index,answer,clue_1,clue_2,clue_3,clue_4,clue_5,clue_6,clue_7,clue_8,clue_9,clue_10
27736,27736,STITCH,Bit of embroidery,Bit of needlework,Do some needlepoint,Embroider,Funny sort,"Hilarious person, colloquially","Hilarious person, in slang",Part of a baseball's seam,Part of a baseball's seam,Part of a baseball's seam


In [219]:
clues_clean.to_csv("clues_clean.csv")